<div style="background-color: #ffffff; color: #000000; padding: 10px;">
<div style="display: flex; justify-content: space-between; align-items: center; background-color: #ffffff; color: #000000; padding: 10px;">
    <img src="../00_aisc/img/logo_aisc_150dpi.png" height="80" style="margin-right: auto;" alt="Logo of the AI Service Center Berlin-Brandenburg.">
    <img src="../00_aisc/img/logo_bmftr_de.png" height="150" style="margin-left: auto;" alt="Logo of the German Federal Ministry of Research, Technology and Space: Gefoerdert vom Bundesministerium fuer Forschung, Technologie und Raumfahrt.">
</div>
<h1> Workshop: Rapid Prototyping via Vibe Coding
<h2> Part 1 &mdash; Claude Code Foundations
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
    <h2> Part 1 &mdash; Claude Code Foundations</h2>
</div>

Claude Code is an **agentic coding environment**. Unlike a chat interface, it reads files, runs commands, makes changes, and autonomously works through problems while you watch, redirect, or step away entirely.

This changes the workflow. Instead of writing code and asking Claude to review it, you describe what you want and Claude explores, plans, and implements.

This notebook covers the foundational knobs you need in order to drive it well:

1. Install and first run.
2. The single constraint behind every best practice: the **context window**.
3. Giving Claude a way to **verify its work** &mdash; the highest-leverage thing you can do.
4. The **Explore &rarr; Plan &rarr; Implement &rarr; Commit** workflow.
5. Providing **specific context** in your prompts.

Configuration (`CLAUDE.md`, permissions, hooks, skills, subagents, plugins) lives in the next notebook, [`w1_02_configure_your_environment.ipynb`](w1_02_configure_your_environment.ipynb).

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. Install and first run</h3>
</div>

Claude Code ships as an npm package. The install is global so the `claude` command is on your `PATH`.

```bash
npm install -g @anthropic-ai/claude-code
```

Verify:

```bash
claude --version
```

#### First session

Run `claude` inside any project directory:

```bash
cd ~/projects/demo-app
claude
```

The first invocation prompts you to log in via your Anthropic account. After that, sessions open instantly.

#### The core slash commands (know these by heart)

| Command | What it does |
|---------|--------------|
| `/help` | Lists all slash commands available in this session. |
| `/status` | Shows the current model, permissions, context usage, and MCP state. |
| `/model` | Switches model (Opus / Sonnet / Haiku). |
| `/clear` | Resets context for a fresh task. Use between unrelated workstreams. |
| `/rewind` | Opens the checkpoint menu (restore conversation, code, or both). |
| `/init` | Generates a starter `CLAUDE.md` (covered in the next notebook). |

#### Turn-taking

- Hit **Enter** to send.
- Press **Esc** while Claude is working to interrupt without losing context.
- Press **Esc twice** (or run `/rewind`) to roll back to a previous checkpoint.
- Press **Shift+Tab** to cycle through permission modes (Normal / Plan / Auto).

#### Try it yourself

```text
in the current directory, list every file and its role in two words or fewer
```

Claude will use `ls`/`tree` and `Read` tools to explore, then summarise. You'll see it thinking, running tools, and streaming back a table.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. The one constraint behind everything: the context window</h3>
</div>

Claude's **context window** holds your entire conversation: every message, every file Claude reads, every command output, the system prompt, and your `CLAUDE.md`. A single debugging session can consume tens of thousands of tokens.

Performance degrades as the window fills. Claude starts "forgetting" earlier instructions and making more mistakes. **The context window is the most important resource to manage &mdash; everything else in this notebook is a strategy for managing it.**

#### How to watch it

Run `/status` any time to see how full the window is. On a long session, it's normal to watch the percentage climb &mdash; that's your cue to consider `/clear` or `/compact`.

#### Mental model

Think of each of these as taking a bite out of the window:

- The system prompt and tool definitions: fixed cost, paid once.
- Your `CLAUDE.md` files: re-read every turn &mdash; keep them tight.
- Every `Read` on a file: pulls the whole file in.
- Every Bash command's stdout: pulls the whole output in. `cat` on a log file is a disaster.
- Every tool result (MCP, web fetches, search results): same.

The best-practices document summarises this as: *"A single debugging session or codebase exploration might generate and consume tens of thousands of tokens."*

#### Why the rest of this notebook exists

Every upcoming best practice &mdash; verification anchors, Plan Mode, specific prompts, `CLAUDE.md` hygiene, permissions, skills, subagents &mdash; is ultimately a way of keeping the useful signal in the window and the noise out.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Give Claude a way to verify its work</h3>
</div>

This is the single highest-leverage thing you can do.

Claude performs dramatically better when it can check itself &mdash; run tests, compare screenshots, validate outputs. Without clear success criteria, it may produce something that looks right but doesn't actually work, and **you become the only feedback loop**. Every mistake then requires your attention.

#### Before / After

| Strategy | Before | After |
|----------|--------|-------|
| **Provide verification criteria** | *"implement a function that validates email addresses"* | *"write a `validate_email` function. Test cases: `user@example.com` &rarr; True, `invalid` &rarr; False, `user@.com` &rarr; False. Run the tests after implementing."* |
| **Verify UI changes visually** | *"make the dashboard look better"* | *"[paste screenshot] implement this design. Screenshot the result and compare it to the original. List differences and fix them."* |
| **Address root causes, not symptoms** | *"the build is failing"* | *"the build fails with this error: `[paste error]`. Fix it and verify the build succeeds. Address the root cause, don't suppress the error."* |

#### What verification can be

- A **test suite** (`pytest`, `vitest`, `cargo test`, ...).
- A **linter or type checker** (`mypy`, `ruff`, `tsc --noEmit`).
- A **Bash check** that greps for a pattern or exits non-zero.
- A **screenshot compare** via the Claude in Chrome extension (`claude.ai/chrome`).

Invest in making your verification rock-solid. If the test suite is flaky, Claude will retry indefinitely; if the lint rule is wrong, Claude will propagate the wrong style.

#### Try it yourself

Paste this into Claude Code inside a project that has a test runner:

```text
add a function `is_palindrome(s: str) -> bool` to src/strings.py.
- empty string -> True
- "racecar" -> True
- "hello" -> False
- "RaceCar" -> True  (case-insensitive)
write tests in tests/test_strings.py and run them. iterate until they pass.
```

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>4. Explore &rarr; Plan &rarr; Implement &rarr; Commit</h3>
</div>

Letting Claude jump straight to coding tends to solve the wrong problem. Separate research from execution using **Plan Mode**.

Enter Plan Mode by pressing **Shift+Tab** until the banner shows `plan mode`. While in Plan Mode, Claude reads files and reasons, but **cannot edit files or run non-read-only commands**.

#### The four phases

##### 1. Explore

In Plan Mode. Claude reads files and answers questions without making changes.

```text
Plan Mode:
read /src/auth and understand how we handle sessions and login.
also look at how we manage environment variables for secrets.
```

##### 2. Plan

Still in Plan Mode. Ask for a detailed implementation plan.

```text
Plan Mode:
I want to add Google OAuth. what files need to change?
what's the session flow? create a plan.
```

Press **Ctrl+G** to open the generated plan in your `$EDITOR` so you can edit it directly before proceeding.

##### 3. Implement

Switch back to Normal Mode (`Shift+Tab` again) and let Claude code.

```text
Normal Mode:
implement the OAuth flow from your plan. write tests for the
callback handler, run the test suite and fix any failures.
```

##### 4. Commit

```text
commit with a descriptive message and open a PR
```

#### When to skip planning

Plan Mode adds overhead. For **clearly scoped, small** changes &mdash; fix a typo, add a log line, rename a variable &mdash; ask Claude to do it directly in Normal Mode.

> **Rule of thumb:** if you could describe the diff in one sentence, skip the plan.

Plan Mode earns its keep when the approach is uncertain, the change spans multiple files, or you're unfamiliar with the code you're modifying.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>5. Provide specific context in your prompts</h3>
</div>

The more precise your instructions, the fewer corrections you'll need. Claude can infer intent, but not read your mind.

| Strategy | Before | After |
|----------|--------|-------|
| **Scope the task** | *"add tests for foo.py"* | *"write a test for `foo.py` covering the edge case where the user is logged out. Avoid mocks."* |
| **Point to sources** | *"why does `ExecutionFactory` have such a weird API?"* | *"look through `ExecutionFactory`'s git history and summarise how its API came to be."* |
| **Reference existing patterns** | *"add a calendar widget"* | *"look at how widgets are implemented on the home page. `HotDogWidget.tsx` is a good example. follow that pattern to implement a new calendar widget. Don't add libraries beyond what's already used."* |
| **Describe the symptom** | *"fix the login bug"* | *"users report that login fails after session timeout. Check the auth flow in `src/auth/`, especially token refresh. Write a failing test that reproduces the issue, then fix it."* |

Vague prompts still have their place when you are *exploring* and want to see how Claude interprets a problem &mdash; e.g., *"what would you improve in this file?"* can surface things you would not have thought to ask.

#### Providing rich content

Five concrete ways to stuff more useful bits into the prompt without bloating it:

- **`@file` references** &mdash; type `@src/auth/session.ts` and Claude reads the file before responding. Works for folders too: `@src/auth/`.
- **Paste screenshots** &mdash; drag, drop, or Cmd/Ctrl+V an image directly into the input. Great for UI bugs.
- **Give URLs** &mdash; Claude can fetch documentation, API references, RFCs. Allowlist the hosts you use often via `/permissions`.
- **Pipe data in** &mdash; `cat error.log | claude -p "summarise the error, suggest a fix"`.
- **Let Claude fetch what it needs** &mdash; *"use `gh pr view 1234` to get the PR, then..."* is better than pasting the PR body yourself.

#### Try it yourself

Open this notebook's companion directory in Claude Code and paste:

```text
read @assets/CLAUDE.example.md and @assets/settings.example.json.
summarise what kind of project they're configured for, and list two things
you'd change before using them in a production repo.
```

You should see Claude read both files, reason about them, and give you a pointed review &mdash; without you having to copy-paste their contents into the prompt.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>6. What's next</h3>
</div>

You now have the **verbs** of Claude Code (install, context-aware prompting, Plan Mode, verification, specificity).

The next notebook &mdash; [`w1_02_configure_your_environment.ipynb`](w1_02_configure_your_environment.ipynb) &mdash; is the **nouns**: the configuration mechanisms (`CLAUDE.md`, permissions, hooks, skills, subagents, plugins, MCP servers) that turn a good terminal session into a repeatable, team-shareable setup.